In [ ]:
train_essays = pd.read_parquet('data/clean_data/train_essays.parquet')
validation_essays = pd.read_parquet('data/clean_data/validation_essays.parquet')

In [ ]:
import pandas as pd
from sklearn.utils import resample

def sample_df(df, col='score', sub=1000, random_state=None):
    """
    Balances the classes in a DataFrame by resampling.
    
    Parameters:
    - df: DataFrame to be resampled.
    - col: The column name in df that contains class labels.
    - random_state: The random state for reproducibility.
    
    Returns:
    - balanced_df: A DataFrame with balanced classes.
    """
    class_counts = df[col].value_counts()
    target_count = max(int(class_counts.median()) - sub, class_counts.min())  # Ensure target_count is positive
    
    balanced_df = pd.DataFrame()

    for class_label in df[col].unique():
        class_subset = df[df[col] == class_label]
        
        if len(class_subset) > target_count:
            # Downsample majority classes
            class_subset = resample(class_subset,
                                    replace=False,
                                    n_samples=target_count,
                                    random_state=random_state)
        else:
            # Upsample minority classes
            class_subset = resample(class_subset,
                                    replace=True,
                                    n_samples=target_count,
                                    random_state=random_state)
        
        balanced_df = pd.concat([balanced_df, class_subset], axis=0)
    
    # Shuffle the DataFrame to mix the classes well
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return balanced_df


# Balance only the training DataFrame
train_essays = sample_df(train_essays, col='score', random_state=42)

validation_essays = sample_df(validation_essays, col='score', sub=0, random_state=42)

In [ ]:
print(train_essays['score'].value_counts())
print(validation_essays['score'].value_counts())

In [ ]:
STATUS = 'Post'

if STATUS == 'Post':
       
       drop_cols = ['full_text', 'lowered', 'clean_text',
        'Pre_tokens', 'Pre_sentences'
       , 'Pre_pos_tags','corrected_text', 'segmented_text',
       'Post_tokens', 'Post_sentences', 'Post_pos_tags']

else:
       drop_cols = ['full_text', 'lowered', 'clean_text',
        'Pre_tokens', 'Pre_sentences', 'Pre_pos_tags',
        'corrected_text', 'segmented_text',]



train_df = train_essays.copy()
val_df = validation_essays.copy()

train_df.drop(columns=drop_cols, inplace= True)
val_df.drop(columns=drop_cols, inplace= True)



In [ ]:
feature_cols = []

for col in train_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle

train_labels = train_df['score']

val_labels = val_df['score']

train_features = train_df[feature_cols]

val_features = val_df[feature_cols]

scaler = StandardScaler()

train_feats_scaled = scaler.fit_transform(train_features)

val_feats_scaled = scaler.transform(val_features)

# Initialize the scaler
target_scaler = StandardScaler()

# Reshape the 1D arrays to 2D arrays with one column each
train_labels_reshaped = np.array(train_labels).reshape(-1, 1)
val_labels_reshaped = np.array(val_labels).reshape(-1, 1)

# Scale the training and validation target variables
train_target_scaled = target_scaler.fit_transform(train_labels_reshaped)
val_target_scaled = target_scaler.transform(val_labels_reshaped)

# Save the scaler object for future use
with open('data/scalers/target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)


with open('data/scalers/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
scaler_path = 'data/scalers/target_scaler.pkl'

# Check if the file has been written correctly and is not empty
import os

if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

In [ ]:
import tensorflow as tf

train_reg = tf.data.Dataset.from_tensor_slices((train_feats_scaled, 
                                                train_target_scaled)).shuffle(len(train_feats_scaled)).batch(32)

val_reg = tf.data.Dataset.from_tensor_slices((val_feats_scaled, 
                                              val_target_scaled)).batch(32)


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import schedules, AdamW, RMSprop

def build_regression_model(hp, input_shape=(train_feats_scaled.shape[1],)):
    
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)  # number of hidden layers
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)  # neurons in each hidden layer
    l2_reg = hp.Float("l2_reg", min_value=1e-6, max_value=1e-2, sampling="log")  # L2 regularization
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")  # learning rate
    
    optimizer_choice = hp.Choice("optimizer_choice", ['adam', 'sgd', 'RMSprop', 'Adagrad', 'Adadelta', 'Nadam', 'Ftrl', 'L-BFGS'])
    
    # Learning rate schedulers
    lr_schedule = schedules.ExponentialDecay(initial_learning_rate=learning_rate,
                                             decay_steps=10000,
                                             decay_rate=0.9)
    
    if optimizer_choice == 'adam':
        optimizer = AdamW(learning_rate=lr_schedule)
    elif optimizer_choice == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)
    elif optimizer_choice == 'RMSprop':
        optimizer = RMSprop(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adagrad':
        optimizer = tf.keras.optimizers.Adagrad(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adadelta':
        optimizer = tf.keras.optimizers.Adadelta(learning_rate=lr_schedule)
    elif optimizer_choice == 'Nadam':
        optimizer = tf.keras.optimizers.Nadam(learning_rate=lr_schedule)
    else:
        # Ftrl will be the default optimizer
        optimizer = tf.keras.optimizers.Ftrl(learning_rate=lr_schedule)

    
    model = tf.keras.models.Sequential()
    
    model.add(tf.keras.Input(shape=input_shape))  # Explicit input layer

    model.add(tf.keras.layers.Dense(hp.Int('input_units', min_value=32, max_value=512, step=32), 
                                    activation='relu'))
    
    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.05)  # dropout rate
    
    # Adding Batch Normalization and dropout for each hidden layer
    for _ in range(n_hidden):
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dense(n_neurons, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
        model.add(tf.keras.layers.Dropout(rate=dropout_rate))
    
    model.add(tf.keras.layers.Dense(1))  # Output layer
    
        # Then, you can add it to your model's compile method as a metric
    model.compile(optimizer=optimizer,  # Use the dynamic optimizer
              loss=tf.keras.losses.Huber(),
              metrics=['mse', 'mae'])



    # Learning rate reduction callback
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)

    return model

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import keras_tuner as kt

model_checkpoint = ModelCheckpoint('data/models/best_regression_model_epoch.keras', 
                                   save_best_only=True, monitor='val_loss', mode='min')

early_stopping = EarlyStopping(monitor='val_loss', patience=5, 
                               restore_best_weights=True)

call_backs = [model_checkpoint, early_stopping]


reg_tuner = kt.BayesianOptimization(
    build_regression_model,
    objective='val_loss',
    max_trials=10,
    num_initial_points= 4,
    seed=1,
    overwrite=True,
    directory='data/models/tuner_reg',
    project_name='regresion',
)


# hyperband model 

# reg_tuner = kt.Hyperband(
#     build_regression_model,
#     objective='val_loss',
#     max_epochs=1000,
#     factor=3,
#     seed=1,
#     overwrite=True,
#     directory='tuner_reg',
#     project_name='regresion',
    
# )



reg_tuner.search(train_reg, epochs=1000, 
             validation_data= val_reg, callbacks = call_backs, verbose=2)

best_regression_model = reg_tuner.get_best_models(num_models=1)[0]


best_regression_model.save('data/models/best_regression_model.keras')

In [ ]:
reg_results = best_regression_model.evaluate(val_reg)


print(reg_results)

In [ ]:
scaled_predictions = best_regression_model.predict(val_feats_scaled)

unscaled_predictions = target_scaler.inverse_transform(scaled_predictions)

rounded_predictions = np.round(unscaled_predictions).flatten()  # Round to nearest integer and flatten if necessary
clipped_predictions = np.clip(rounded_predictions, 1, 6) # Clip to ensure within 1-6 range
clipped_predictions = clipped_predictions.astype(int)

# Now, `clipped_predictions` contains the integer score predictions


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming `clipped_predictions` are your model predictions
# and `actual_scores` (or `val_labels`) are the ground truth scores for the test set

# Calculate the absolute errors
errors = np.abs(clipped_predictions - val_labels)

# You can scale the errors to ensure the sizes are visible and meaningful on the plot
# Adjust the scaling factor as needed to get a clear visual representation
size = errors * 10  # Example scaling factor

# Scatter plot of actual vs. predicted scores
plt.figure(figsize=(10, 6))
plt.scatter(val_labels, clipped_predictions, s=size, alpha=0.6)  # Use size for marker size
plt.title('Actual vs. Predicted Scores')
plt.xlabel('Actual Scores')
plt.ylabel('Predicted Scores')
plt.plot([1, 6], [1, 6], 'r--')  # Diagonal line representing perfect predictions
plt.colorbar(label='Absolute Error')  # Optionally add a colorbar to indicate the size meaning
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming `clipped_predictions` are your model predictions
# and `actual_scores` are the ground truth scores for the test set

# Scatter plot of actual vs. predicted scores
plt.figure(figsize=(10, 6))
plt.scatter(val_labels, clipped_predictions, alpha=0.6)
plt.title('Actual vs. Predicted Scores')
plt.xlabel('Actual Scores')
plt.ylabel('Predicted Scores')
plt.plot([1, 6], [1, 6], 'r--')  # Diagonal line representing perfect predictions
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming `clipped_predictions` contains the regression model's predictions rounded and clipped to 1-6
# And `actual_scores` contains the true scores

# Compute the confusion matrix
cm = confusion_matrix(val_labels, clipped_predictions)

# Visualize the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=range(1, 7), yticklabels=range(1, 7))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Scores')
plt.ylabel('Actual Scores')
plt.show()


In [ ]:
print(sorted(val_labels.unique()))
print(sorted(pd.Series(clipped_predictions).unique()))

In [ ]:
# Example usage

actuals = val_labels
predictions = clipped_predictions


from sklearn.metrics import cohen_kappa_score

# Calculate the Quadratic Weighted Kappa
qwk_score = cohen_kappa_score(actuals, predictions, weights='quadratic')

print("Quadratic Weighted Kappa score:", qwk_score)
